# Feature Engineering and Selection

Interactions capture joint effects, e.g. price × promotion. Polynomial expansions increase model flexibility and overfitting risk. Selection and scaling must be fitted inside each CV fold to avoid leakage.

**Synthetic educational data.**

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def locate(filename):
    paths=[pathlib.Path('data')/filename,pathlib.Path('../data')/filename,pathlib.Path(filename),pathlib.Path('/content')/filename]
    for path in paths:
        if path.exists(): return path
    raise FileNotFoundError(f'Upload {filename} or run notebook from repository root')

churn=pd.read_csv(locate('customer_churn_synthetic.csv'))
sales=pd.read_csv(locate('store_sales_synthetic.csv'))
segments=pd.read_csv(locate('customer_segments_synthetic.csv'))
print('Loaded:',len(churn),'churn rows,',len(sales),'sales rows,',len(segments),'segment rows')

In [ ]:
display(churn.head(3)); print('Class balance:'); print(churn['churn'].value_counts(normalize=True).round(3))

## Core demonstration
Run the model below, then interpret the output in business terms.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
X=sales.drop(columns='weekly_sales'); y=sales.weekly_sales
model=make_pipeline(PolynomialFeatures(degree=2,include_bias=False),StandardScaler(),Ridge(alpha=10))
scores=cross_val_score(model,X,y,cv=5,scoring='neg_mean_absolute_error')
print('5-fold MAE:',round(-scores.mean(),2))

## Visualization and interpretation
A simple distribution plot helps reveal the range and shape of an important feature.

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
churn['tenure_months'].hist(bins=25,ax=ax)
ax.set(title='Synthetic feature distribution',xlabel='Tenure feature',ylabel='Count')
plt.tight_layout(); plt.show()

## Questions
1. What baseline should this model beat?
2. What would be a costly error?
3. How could leakage enter this workflow?
4. What additional validation would you require before deployment?